In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml
config = yaml.safe_load(open('../config.yaml', 'r'))

import os
os.environ["CUDA_VISIBLE_DEVICES"] = '8'
import torch
import torch.nn.functional as F

from src.model.models_cnn import EDSR
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random


In [ ]:
model = EDSR(n_resblocks=5, n_feats=32)
model.load_state_dict(torch.load('../model/cnn/cnn_weights_20250626.pt', weights_only=True))

In [ ]:
dataset = dataset_sr()
loss = torch.nn.MSELoss()

In [ ]:
i = random.sample(range(40000), 1)[0]

scale_factor = 4
sr_factor = 4
fig, axes = plt.subplots(3, 3, figsize=(12,12))

# plot cuts at this z level
z_level = random.sample(range(128), 1)[0]
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor

for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
hr_state, lr_state_tensor, _, _ = dataset[i]

with torch.no_grad():
    sr_state = torch.squeeze(model(torch.unsqueeze(lr_state_tensor, 0)),0)
print(loss(sr_state, hr_state))
sr_state = sr_state.cpu().detach().numpy()
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()
figures = []
figures.append(axes[0,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], cmap = 'jet'))
axes[0,0].set_ylabel(f"{i}")
figures.append(
    axes[0, 1].imshow(
        np.sqrt(
            lr_state[1, :, :, z_level_reduced] ** 2
            + lr_state[2, :, :, z_level_reduced] ** 2
        ).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
figures.append(
    axes[0, 2].imshow(
        lr_state[4, :, :, z_level_reduced].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)

figures.append(
    axes[1, 0].imshow(
        hr_state[0, :, :, z_level].T, origin="lower", extent=[0, 1, 0, 1], cmap="jet"
    )
)
figures.append(
    axes[1, 1].imshow(
        np.sqrt(hr_state[1, :, :, z_level] ** 2 + hr_state[2, :, :, z_level] ** 2).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
figures.append(axes[1,2].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], cmap = 'jet'))

figures.append(axes[2,0].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], cmap = 'jet'))
figures.append(axes[2,1].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2 + sr_state[3, :, :, z_level_sr]**2 ).T, origin = "lower", extent = [0, 1, 0, 1], cmap = 'jet'))
figures.append(axes[2,2].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1] , cmap = 'jet'))

for figs in figures:
    fig.colorbar(figs)
axes[0,0].set_title("Density")
axes[0,1].set_title("Velocity") 
axes[0,2].set_title("Pressure")

In [ ]:
print(np.sum((hr_state[4] < 0)))
print(np.sum((sr_state[4] < 0)))


In [ ]:
print(np.sum((lr_state < 0.010) & (lr_state > -0.01)))
